<a href="https://colab.research.google.com/github/sam098761212-glitch/Leeseungsoo-AI_CLASS/blob/main/abalone_DL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, Sequential
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, accuracy_score

tf.keras.utils.set_random_seed(42)
if not os.path.exists("abalone.csv"):
    from google.colab import files
    files.upload()

df = pd.read_csv("abalone.csv")
X = pd.get_dummies(df.drop(columns=["id", "Rings"]), columns=["Sex"], dtype="float32").to_numpy(dtype="float32")
y_reg = df["Rings"].to_numpy(dtype="float32")
# Rings 8 이하=0, 9~10=1, 11 이상=2로 분류
y_cls = np.where(y_reg <= 8, 0, np.where(y_reg <= 10, 1, 2))

X_train, X_test, yr_train, yr_test, yc_train, yc_test = train_test_split(
    X, y_reg, y_cls, test_size=0.2, random_state=42, stratify=y_cls
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype("float32")
X_test = scaler.transform(X_test).astype("float32")
stop = tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)


In [2]:
regression_model = Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(64, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(1)
])
regression_model.compile(optimizer="adam", loss="mse", metrics=["mae"])
regression_model.fit(
    X_train, yr_train, epochs=100, batch_size=32, validation_split=0.2,
    callbacks=[stop], verbose=1
)
reg_pred = regression_model.predict(X_test, verbose=0).ravel()
print("Abalone 회귀 MAE:", mean_absolute_error(yr_test, reg_pred))

Epoch 1/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 64.5201 - mae: 7.1134 - val_loss: 23.4257 - val_mae: 3.7111
Epoch 2/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 13.9657 - mae: 2.7501 - val_loss: 11.9712 - val_mae: 2.0958
Epoch 3/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 6.5864 - mae: 1.7900 - val_loss: 10.9163 - val_mae: 1.7709
Epoch 4/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 5.6131 - mae: 1.6731 - val_loss: 10.6537 - val_mae: 1.7207
Epoch 5/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 5.3253 - mae: 1.6314 - val_loss: 10.3051 - val_mae: 1.6901
Epoch 6/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 5.1573 - mae: 1.6058 - val_loss: 9.9376 - val_mae: 1.6690
Epoch 7/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 5.0439 - mae: 1.5885 - val_loss: 9.5928 - val_mae: 1.6524
Epoch 8/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 4.9565 - mae: 1.5747 - val_loss: 9.2740 - val_mae: 1.6402
Epoch 9/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss

In [3]:
classification_model = Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(64, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(3, activation="softmax")
])
classification_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
classification_model.fit(
    X_train, yc_train, epochs=100, batch_size=32, validation_split=0.2,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)], verbose=1
)
cls_pred = classification_model.predict(X_test, verbose=0).argmax(axis=1)
print("Abalone 분류 정확도:", accuracy_score(yc_test, cls_pred))


Epoch 1/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.5273 - loss: 0.9082 - val_accuracy: 0.6158 - val_loss: 0.7796
Epoch 2/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6108 - loss: 0.8029 - val_accuracy: 0.6577 - val_loss: 0.7560
Epoch 3/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6355 - loss: 0.7762 - val_accuracy: 0.6607 - val_loss: 0.7419
Epoch 4/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.6445 - loss: 0.7551 - val_accuracy: 0.6607 - val_loss: 0.7317
Epoch 5/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6538 - loss: 0.7379 - val_accuracy: 0.6592 - val_loss: 0.7282
Epoch 6/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6609 - loss: 0.7265 - val_accuracy: 0.6622 - val_loss: 0.7280
Epoch 7/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6635 - loss: 0.7189 - val_accuracy: 0.6682 - val_loss: 0.7271
Epoch 8/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6647 - loss: 0.7136 - val_accuracy: 0.6712 - v